In [ ]:
import os
import numpy as np
import pandas as pd
from glob import glob
from tqdm import tqdm
from pathlib import Path

### MERGE to SPCS

In [ ]:
data_path = 'PATH_TO_DATA_DIRECTORY'  # This should contain 'umi_counts', 'features', and 'tissue_positions' directories
out_path = 'PATH_TO_OUTPUT_DIRECTORY'  # Replace with your actual output directory

Path(os.path.join(out_path, 'counts')).mkdir(parents=True, exist_ok=True)
Path(os.path.join(out_path, 'coords')).mkdir(parents=True, exist_ok=True)

counts_paths = glob(os.path.join(data_path, 'umi_counts', '*'))
features_paths = glob(os.path.join(data_path, 'features', '*'))
tissue_positions_paths = glob(os.path.join(data_path, 'tissue_positions', '*'))

counts_paths.sort()
features_paths.sort()
tissue_positions_paths.sort()


In [3]:
# Check if the file names are the same
for counts_path, features_path, tissue_positions_path in zip(counts_paths, features_paths, tissue_positions_paths):
    assert Path(counts_path).stem == Path(features_path).stem == Path(tissue_positions_path).stem

In [ ]:
for counts_path, features_path, tissue_positions_path in tqdm(zip(counts_paths, features_paths, tissue_positions_paths)):
    # Read the data
    counts = np.load(counts_path)
    features = pd.read_csv(features_path, header=None)
    tissue_positions = pd.read_csv(tissue_positions_path, header=None)
    
    # Only keep the spots that are in the tissue
    tissue_positions = tissue_positions[tissue_positions[1]==1]
    
    counts_df = pd.DataFrame(counts, columns=features[0].tolist())
    counts_df.index = tissue_positions[0].tolist()
    
    # If there are duplicate column names, keep the one that has highest sum
    counts_df = counts_df.groupby(counts_df.columns, axis=1).sum()
    
    # Transpose the counts so that the spots are in the columns
    counts_df = counts_df.T
    
    # Save the counts as text file
    counts_save_path = os.path.join(out_path, 'counts', Path(counts_path).stem + '.txt')
    counts_df.to_csv(counts_save_path, sep=',', index=True)
    
    # Open the counts file and delete the first character from the first line and save again
    with open(counts_save_path, 'r') as f:
        lines = f.readlines()
        lines[0] = lines[0][1:]
        
    with open(counts_save_path, 'w') as f:
        f.writelines(lines)
        
    # Save the coordinates as text file
    coords_save_path = os.path.join(out_path, 'coords', Path(counts_path).stem + '.txt')
    coords_df = tissue_positions.drop(columns=[1,4,5])
    coords_df.columns = ['', 'coord1', 'coord2']
    
    coords_df.to_csv(coords_save_path, sep=',', index=False)
    
    # Open the counts file and delete the first character from the first line and save again
    with open(coords_save_path, 'r') as f:
        lines = f.readlines()
        lines[0] = lines[0][1:]
        
    with open(coords_save_path, 'w') as f:
        f.writelines(lines)

### SPCS to MERGE

In [ ]:
out_path = 'PATH_TO_OUTPUT_DIRECTORY'  # Replace with your actual output directory

counts_path = os.path.join(out_path, 'counts_spcs')
features_path = os.path.join(out_path, 'features')

Path(counts_path).mkdir(parents=True, exist_ok=True)
Path(features_path).mkdir(parents=True, exist_ok=True)

smoothed_counts_paths = glob('PATH_TO_SMOOTHED_COUNTS_DIRECTORY/*') # Replace with the actual path to your smoothed counts files generated after running SPCS
smoothed_counts_paths.sort()

In [ ]:
for smoothed_counts_path in smoothed_counts_paths:
    smoothed_counts = pd.read_csv(smoothed_counts_path, sep=',', index_col=0).T
    smoothed_counts.index = [item.replace('.', '-') for item in smoothed_counts.index]
    
    # Save counts as npy file
    np.save(os.path.join(counts_path, Path(smoothed_counts_path).stem + '.npy'), smoothed_counts.values)
    
    # Save genes as csv file
    cols = pd.DataFrame(smoothed_counts.columns)
    cols.to_csv(os.path.join(features_path, Path(smoothed_counts_path).stem + '.csv'), index=False, header=False)